<a href="https://colab.research.google.com/github/Lobnaait/SEARCH_Lobna_Tsetline_CMRI/blob/main/Tsetline_ImagesDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyTsetlinMachine

  Preparing metadata (setup.py) ... done
  Created wheel for pyTsetlinMachine: filename=pytsetlinmachine-0.6.6-cp313-cp313-linux_x86_64.whl size=59793 sha256=704c9d43c13bdf34ad452fc19170912afb36411ea42ebdac184d38dcd7a4c958
  Stored in directory: /root/.cache/pip/wheels/a9/66/98/535c2cc844fdb6fc12f31feefbaa6a33222b1378914098f498
Successfully built pyTsetlinMachine


In [2]:
#IMPORT LIBRARIES AND DATASET
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pyTsetlinMachine.tm import MultiClassTsetlinMachine

dataset = load_digits()  # Load the images dataset

In [26]:
# Divide the dataset into training set and test set for features (X) and target (y)
from sklearn.model_selection import train_test_split
X = dataset.images # input images
y = dataset.target # output labels
X_train, X_test, y_train, y_test = train_test_split(dataset.images, dataset.target, test_size = 0.25)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20)


# How many tresholds are better to use?
Since the treshold is an hyperparameter I have to perform the validation step to obtain the optiman treshold.
For validation we use part of the training data

In [4]:
# CHOOSE THE TRESHOLDS: Based on the number of treshold divide the dataset into tresholds
def compute_thresholds(X, n_thresholds):
  # Extract non-zero pixel
  non_zero_pixels = X[X > 0]
  percentile_values = np.linspace(0, 100, int(n_thresholds) + 2)[1:-1] # Exclude 0 and 100
  thresholds = np.percentile(non_zero_pixels, percentile_values)
  return np.unique(thresholds)

In [13]:
def booleanise(X, thresholds):
  # Flatten each image
  X_flat = X.reshape(X.shape[0], -1)
  X_bool = (X_flat[:, :, None] > thresholds[None, None, :]) #Compare every pixel with every threshold
  #convert to binary
  X_bool = X_bool.astype(np.uint32)
  X_bool = X_bool.reshape(X_bool.shape[0], -1)
  return X_bool

In [27]:
# VALIDATION: Test how many tresholds are optimal to use?
threshold_candidates = [1,10,20]
results = []

for n_thresholds in threshold_candidates:

    thresholds = compute_thresholds(X_train, n_thresholds)
    X_train_bool = booleanise(X_train, thresholds)
    X_val_bool = booleanise(X_val, thresholds)

    # model
    tm = MultiClassTsetlinMachine(number_of_clauses=1000, T=50, s=5.0)
    tm.fit(X_train_bool, y_train, epochs=100)

    val_accuracy = accuracy_score(y_val, tm.predict(X_val_bool))

    # Corrected: Append a dictionary to results
    results.append({
        "n_thresholds": n_thresholds,
        "accuracy": val_accuracy
    })

In [29]:
print(results)

[{'n_thresholds': 1, 'accuracy': 0.9444444444444444}, {'n_thresholds': 10, 'accuracy': 0.9703703703703703}, {'n_thresholds': 20, 'accuracy': 0.9740740740740741}]
